# 📝 판다스 데이터 분석 기초 과제 LV1 정답 — DataFrame 조회·선택·정제 (강사용)

각 문제의 **모범답안 + 해설(접근법·흔한 실수·대안)** 입니다. 스스로 먼저 풀어 본 뒤 비교해 보세요.

## 1. 데이터 불러오기
**배경**: 분석의 첫걸음은 파일을 표(DataFrame)로 불러와 크기와 열 이름을 확인하는 일입니다.

**요구사항**:
- `pandas` 를 `pd` 라는 이름으로 불러오세요.
- `data/used_cars.csv` 를 읽어 `df` 에 담으세요.
- `df` 의 모양(행·열 개수)과 열 이름 목록을 확인하세요.

**예시**
```
df.shape        →  (50, 7)
list(df.columns) →  ['모델', '연식', ...]
```

<details><summary>힌트</summary>

```text
접근방법:
- 판다스로 CSV 파일을 읽어 표로 만든 뒤, 크기와 열 이름을 확인한다.

세부구현:
1. 판다스를 pd 로 불러온다
2. read_csv 로 파일을 읽어 df 에 담는다
3. shape 로 (행, 열) 을, columns 로 열 이름을 확인한다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
print(df.shape)
print(list(df.columns))

In [ ]:
# [자가채점]
assert df.shape == (50, 7), "행 50, 열 7 이어야 해요"
assert list(df.columns) == ['모델', '연식', '주행거리', '가격만원', '연료', '색상', '사고여부']
print("✅ 문제1 통과!")

### 해설 — 문제 1
- **접근법**: `pd.read_csv` 는 CSV 를 DataFrame 으로 만들어 줍니다. `shape` 는 `(행, 열)` 튜플, `columns` 는 열 이름들을 담은 Index 예요.
- **흔한 실수**: 파일 경로. 과제 노트북은 `data/`, 정답 노트북은 `../../day06_판다스_기초/data/` 를 씁니다(폴더 위치가 달라서요).
- **대안**: 열 이름만 빠르게 보려면 `df.columns.tolist()` 도 같은 결과입니다.

## 2. 데이터 첫인상 살펴보기 (서술형)
**배경**: 본격 분석 전에 데이터를 눈으로 훑어 **무엇이 이상한지** 감을 잡습니다. 아래 셀을 실행해 출력(앞부분·요약·자료형)을 보고, 무엇을 정제해야 할지 서술하세요.

**요구사항**: 출력을 근거로 아래 세 가지를 찾아 서술 셀에 적으세요.
- 결측값(비어 있는 값)이 있는 열은 무엇이고 몇 개인가?
- 숫자처럼 보이지만 문자열(object)로 읽힌 열은 무엇인가? 왜 그럴까?
- `가격만원` 의 최솟값을 보고 이상해 보이는 점은?

**예시**: `*(색상 열에 결측이 N개 있다 … 처럼 관찰한 사실을 적으세요)*`

<details><summary>힌트</summary>

```text
접근방법:
- info 출력의 Non-Null 개수, describe 의 min/max, 자료형(Dtype)을 근거로 이상한 점을 말로 정리한다.

세부구현:
1. info 에서 각 열의 Non-Null Count 가 전체 행 수보다 작으면 결측이 있는 열
2. Dtype 이 object 인데 숫자여야 할 열은 콤마 같은 문자가 섞여 문자열로 읽힌 것
3. describe 의 min 이 상식 밖(예: 가격 0)이면 이상치 후보
```

</details>

In [ ]:
# [제공 코드]
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
print(df.head())
print()
df.info()
print()
print(df[['연식', '가격만원']].describe())

**모범 서술**

- **결측값**: `색상` 열의 Non-Null Count 가 46 으로, 전체 50행보다 4 적습니다. 즉 색상에 결측이 **4개** 있습니다. 나머지 열은 50개가 모두 채워져 있습니다.
- **문자열로 읽힌 숫자 열**: `주행거리` 의 Dtype 이 `object` 입니다. 값이 `"89,000"` 처럼 **천 단위 콤마가 들어간 문자열**이라 판다스가 숫자로 인식하지 못했습니다. 계산에 쓰려면 콤마를 지우고 정수로 바꿔야 합니다.
- **가격 이상치**: `가격만원` 의 min 이 `0` 입니다. 값이 0원인 매물은 현실적으로 이상하므로 이상치로 의심하고 분석에서 걸러야 합니다. (`주행거리` 에도 `999,999` 같은 비현실적 값이 섞여 있습니다.)

## 3. 열 선택하기
**배경**: 필요한 열만 골라 보는 일은 가장 기본적인 조회입니다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `가격만원` 한 열만 골라 `price_col` 에 담으세요. (결과는 Series)
- `모델`, `가격만원` 두 열을 골라 `two_cols` 에 담으세요. (결과는 DataFrame, 열 순서 그대로)

**예시**
```
price_col.shape  →  (50,)
list(two_cols.columns) →  ['모델', '가격만원']
```

<details><summary>힌트</summary>

```text
접근방법:
- 대괄호 하나로 한 열(Series)을, 대괄호 두 겹(열 이름 리스트)으로 여러 열(DataFrame)을 고른다.

세부구현:
1. 파일을 df 로 불러온다
2. df 에 열 이름 하나를 대괄호로 넣어 price_col 을 만든다
3. df 에 열 이름 두 개를 리스트로 넣어 two_cols 를 만든다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
price_col = df['가격만원']
two_cols = df[['모델', '가격만원']]
print(price_col.shape)
print(two_cols.shape)

In [ ]:
# [자가채점]
assert price_col.shape == (50,), "한 열은 Series (50,) 모양"
assert list(two_cols.columns) == ['모델', '가격만원']
assert two_cols.shape == (50, 2)
print("✅ 문제3 통과!")

### 해설 — 문제 3
- **접근법**: `df['열']` 은 1차원 Series, `df[['열1','열2']]` 는 2차원 DataFrame 입니다. 대괄호 개수가 결과 차원을 정합니다.
- **흔한 실수**: 여러 열을 고를 때 `df['모델','가격만원']` 처럼 리스트를 빼먹으면 에러가 납니다. 반드시 `[[ ... ]]`.
- **대안**: `df.loc[:, ['모델','가격만원']]` 도 같은 결과를 줍니다.

## 4. 열과 행 버리기 (drop)
**배경**: 이번 분석에 필요 없는 열과, 잘못 입력된 행을 덜어 냅니다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `사고여부` 열을 버린 결과를 `df_cols` 에 저장하세요. (`drop(columns=...)`)
- `가격만원` 이 0인 행(잘못 입력된 이상치)을 버린 결과를 `df_rows` 에 저장하세요. (`drop(index=...)`)

**예시**
```
len(df_cols.columns)  →  6   (7개 중 사고여부 1개 뺌)
len(df_rows)          →  49  (50행 중 가격 0인 1행 뺌)
```

<details><summary>힌트</summary>

```text
접근방법:
- 열은 columns=, 행은 index= 로 버린다. 가격이 0인 행의 인덱스를 먼저 찾는다.

세부구현:
1. df.drop(columns=['사고여부']) 를 df_cols 에 저장
2. df['가격만원']==0 인 행의 인덱스를 구해 df.drop(index=그 인덱스) 를 df_rows 에 저장
```

</details>

In [ ]:
df = pd.read_csv("../../day06_판다스_기초/data/used_cars.csv")
df_cols = df.drop(columns=['사고여부'])
df_rows = df.drop(index=df.index[df['가격만원'] == 0])
print(len(df_cols.columns), len(df_rows))

In [ ]:
# [자가채점]
assert len(df_cols.columns) == 6 and '사고여부' not in df_cols.columns
assert len(df_rows) == 49
assert (df_rows['가격만원'] == 0).sum() == 0, "가격만원 0인 이상치 행이 남아있으면 안 돼요"
print("✅ 문제4 통과!")

### 해설 — 문제 4
- **접근법**: 열은 `drop(columns=[...])`, 행은 `drop(index=[...])` 로 버립니다. 둘 다 **원본은 그대로 두고 새 표를 돌려줍니다**(원본을 바꾸려면 다시 대입).
- **흔한 실수**: `drop('사고여부')` 만 쓰면 축이 애매합니다 — 열이면 `columns=`, 행이면 `index=` 를 명시하세요.
- **대안**: 여러 열은 `drop(columns=['a','b'])` 처럼 리스트로 한 번에 버립니다.

## 5. 컬럼 이름 바꾸기 (rename)
**배경**: `가격만원`·`사고여부` 처럼 긴 열 이름을 짧고 깔끔하게 바꿉니다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `가격만원` 을 `가격` 으로, `사고여부` 를 `사고` 로 바꾼 결과를 `df2` 에 저장하세요. (`rename(columns=...)`)

**예시**
```
list(df2.columns)  →  ['모델', '연식', '주행거리', '가격', '연료', '색상', '사고']
```

<details><summary>힌트</summary>

```text
접근방법:
- rename 의 columns 에 {옛이름: 새이름} 딕셔너리를 준다.

세부구현:
1. df.rename(columns={'가격만원':'가격', '사고여부':'사고'}) 를 df2 에 저장
```

</details>

In [ ]:
df = pd.read_csv("../../day06_판다스_기초/data/used_cars.csv")
df2 = df.rename(columns={'가격만원': '가격', '사고여부': '사고'})
print(list(df2.columns))

In [ ]:
# [자가채점]
assert list(df2.columns) == ['모델', '연식', '주행거리', '가격', '연료', '색상', '사고']
print("✅ 문제5 통과!")

### 해설 — 문제 5
- **접근법**: `rename(columns={{옛이름: 새이름}})` — 바꿀 열만 딕셔너리에 적으면 나머지는 그대로 둡니다.
- **흔한 실수**: 딕셔너리 방향을 거꾸로(새이름: 옛이름) 쓰면 안 됩니다. **옛이름이 키**입니다.
- **대안**: 모든 열을 한 번에 바꾸려면 `df.columns = [...]` 로 새 목록을 통째로 대입할 수도 있습니다.

## 6. loc 로 라벨 기반 선택
**배경**: `loc` 은 **행 라벨(인덱스)과 열 이름**으로 값을 고릅니다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `loc` 으로 인덱스 0~4번 행의 `연식`, `가격만원` 두 열을 골라 `part` 에 담으세요.
- `loc` 으로 인덱스 3번 행의 `연료` 값을 골라 `fuel3` 에 담으세요.

**예시**
```
part.shape  →  (5, 2)
fuel3       →  '디젤'
```

<details><summary>힌트</summary>

```text
접근방법:
- loc 은 [행 라벨, 열 이름] 순서로 넣는다. 라벨 슬라이스 0:4 는 끝(4)을 포함한다.

세부구현:
1. 파일을 df 로 불러온다
2. df.loc[0:4, ['연식','가격만원']] 로 part 를 만든다 (loc 슬라이스는 끝 포함)
3. df.loc[3, '연료'] 로 fuel3 를 만든다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
part = df.loc[0:4, ['연식', '가격만원']]
fuel3 = df.loc[3, '연료']
print(part)
print(fuel3)

In [ ]:
# [자가채점]
assert part.shape == (5, 2), "0~4번은 5개 행 (loc 은 끝 포함)"
assert list(part.columns) == ['연식', '가격만원']
assert fuel3 == '디젤'
print("✅ 문제6 통과!")

### 해설 — 문제 6
- **접근법**: `loc` 은 이름표(라벨)로 고릅니다. 라벨 슬라이스 `0:4` 는 파이썬 리스트와 달리 **끝 값 4를 포함**해 5개 행을 줍니다.
- **흔한 실수**: `loc` 과 `iloc` 을 헷갈리기 쉬워요. `loc` 은 라벨, `iloc` 은 위치 번호입니다.
- **대안**: `df.loc[0:4][['연식','가격만원']]` 처럼 나눠 써도 되지만 한 번에 `loc[행, 열]` 이 더 명확합니다.

## 7. iloc 로 위치 기반 선택
**배경**: `iloc` 은 <strong>정수 위치(0부터)</strong>로 값을 고릅니다. 엑셀에서 "몇 번째 칸"을 집는 느낌이에요.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `iloc` 으로 **처음 5개 행, 처음 3개 열**을 골라 `block` 에 담으세요.
- `iloc` 으로 0번 행, 1번 열의 값을 골라 `y0` 에 담으세요.

**예시**
```
block.shape  →  (5, 3)
y0           →  2016   (0번 행의 연식)
```

<details><summary>힌트</summary>

```text
접근방법:
- iloc 은 위치 슬라이스라 :5 는 0~4(끝 미포함)를 뜻한다. [행위치, 열위치] 순서.

세부구현:
1. 파일을 df 로 불러온다
2. df.iloc[:5, :3] 로 block 을 만든다 (위치 슬라이스는 끝 미포함)
3. df.iloc[0, 1] 로 y0 를 만든다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
block = df.iloc[:5, :3]
y0 = df.iloc[0, 1]
print(block)
print(y0)

In [ ]:
# [자가채점]
assert block.shape == (5, 3), "위치 슬라이스 :5, :3"
assert y0 == 2016
print("✅ 문제7 통과!")

### 해설 — 문제 7
- **접근법**: `iloc` 은 순수 위치 번호예요. 파이썬 슬라이스와 똑같이 `:5` 는 0~4까지(끝 미포함)입니다.
- **흔한 실수**: `loc[0:4]` 는 5개(끝 포함), `iloc[0:4]` 는 4개(끝 미포함). 둘의 슬라이스 규칙이 달라요.
- **대안**: 마지막 행은 `df.iloc[-1]` 처럼 음수 위치로도 집을 수 있습니다.

## 8. 조건으로 행 거르기 (단일 조건)
**배경**: "디젤 차만 보고 싶다" 처럼 조건에 맞는 행만 남기는 것이 **불리언 필터**입니다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `연료` 가 `'디젤'` 인 행만 골라 `diesel_df` 에 담으세요.

**예시**
```
len(diesel_df)  →  (디젤 매물 수)
diesel_df['연료'] 는 모두 '디젤'
```

<details><summary>힌트</summary>

```text
접근방법:
- 열에 비교 연산을 걸면 참/거짓 Series 가 나오고, 그걸 df 의 대괄호에 넣으면 참인 행만 남는다.

세부구현:
1. 파일을 df 로 불러온다
2. df['연료'] == '디젤' 로 조건(참/거짓)을 만든다
3. 그 조건을 df[ ... ] 에 넣어 diesel_df 를 만든다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
diesel_df = df[df['연료'] == '디젤']
print(len(diesel_df))

In [ ]:
# [자가채점]
assert len(diesel_df) == 11, "디젤 매물은 11대"
assert (diesel_df['연료'] == '디젤').all()
print("✅ 문제8 통과!")

### 해설 — 문제 8
- **접근법**: `df['연료'] == '디젤'` 은 각 행이 참인지 거짓인지 담은 Series 입니다. `df[조건]` 은 참인 행만 남겨요.
- **흔한 실수**: `df['연료' == '디젤']` 처럼 대괄호 **안에서** 비교하면 안 됩니다. 조건은 대괄호 밖에서 먼저 만들어요.
- **대안**: `df.query("연료 == '디젤'")` 도 같은 결과를 줍니다.

## 9. 두 조건을 함께 (AND)
**배경**: 조건이 둘 이상이면 `&`(그리고), `|`(또는)로 잇습니다. 각 조건은 **괄호로 감싸야** 해요.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `연식` 이 2020년 **이상**이면서 `사고여부` 가 `'무사고'` 인 행만 골라 `recent_safe` 에 담으세요.

**예시**
```
len(recent_safe)  →  (조건을 모두 만족하는 매물 수)
```

<details><summary>힌트</summary>

```text
접근방법:
- 두 조건을 각각 괄호로 감싸고 & 로 잇는다. and/or 가 아니라 &/| 를 쓴다.

세부구현:
1. 파일을 df 로 불러온다
2. (df['연식'] >= 2020) 와 (df['사고여부'] == '무사고') 를 & 로 잇는다
3. 그 조건을 df[ ... ] 에 넣어 recent_safe 를 만든다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
recent_safe = df[(df['연식'] >= 2020) & (df['사고여부'] == '무사고')]
print(len(recent_safe))

In [ ]:
# [자가채점]
assert len(recent_safe) == 12, "2020년 이상 & 무사고 = 12대"
assert (recent_safe['연식'] >= 2020).all()
assert (recent_safe['사고여부'] == '무사고').all()
print("✅ 문제9 통과!")

### 해설 — 문제 9
- **접근법**: 판다스 불리언 필터는 `and`/`or` 대신 `&`/`|` 를 씁니다. 각 조건은 반드시 **괄호**로 감싸요 (연산자 우선순위 때문).
- **흔한 실수**: 괄호를 빼면(`df['연식'] >= 2020 & ...`) 우선순위가 꼬여 에러가 납니다.
- **대안**: `~` 로 조건을 뒤집을 수도 있어요 (예: `~(df['사고여부'] == '유사고')`).

## 10. 목록·범위로 거르기 (isin · between)
**배경**: "가솔린 또는 디젤" 처럼 여러 값 중 하나면 `isin`, "2000~3000 사이" 처럼 범위면 `between` 이 깔끔합니다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `연료` 가 `'가솔린'` 또는 `'디젤'` 인 행을 `isin` 으로 골라 `gd` 에 담으세요.
- `가격만원` 이 2000 이상 3000 이하인 행을 `between` 으로 골라 `mid` 에 담으세요. (`between` 은 양 끝을 포함)

**예시**
```
len(gd)   →  (가솔린·디젤 매물 수)
len(mid)  →  (2000~3000 매물 수)
```

<details><summary>힌트</summary>

```text
접근방법:
- isin 은 값 목록(리스트)을, between 은 하한과 상한을 받는다. 둘 다 참/거짓 Series 를 만든다.

세부구현:
1. 파일을 df 로 불러온다
2. df['연료'].isin([...]) 로 gd 를 고른다
3. df['가격만원'].between(2000, 3000) 로 mid 를 고른다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
gd = df[df['연료'].isin(['가솔린', '디젤'])]
mid = df[df['가격만원'].between(2000, 3000)]
print(len(gd), len(mid))

In [ ]:
# [자가채점]
assert len(gd) == 29, "가솔린 또는 디젤 = 29대"
assert len(mid) == 16, "가격 2000~3000 = 16대"
print("✅ 문제10 통과!")

### 해설 — 문제 10
- **접근법**: `isin([...])` 은 여러 `|` 조건을 한 줄로 줄여 줍니다. `between(a, b)` 는 `(x >= a) & (x <= b)` 와 같아요(양 끝 포함).
- **흔한 실수**: `between` 이 끝값을 포함하는지 헷갈리기 쉬워요. 기본은 **포함**입니다(`inclusive='both'`).
- **대안**: `isin` 없이 `(df['연료']=='가솔린') | (df['연료']=='디젤')` 로도 되지만 값이 많아지면 `isin` 이 깔끔합니다.

## 11. 여러 기준으로 정렬
**배경**: "연식이 오래된 순, 같은 연식이면 비싼 순" 처럼 **여러 기준**으로 줄 세울 수 있습니다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `연식` 은 **오름차순**, 같은 연식이면 `가격만원` 은 **내림차순**으로 정렬해 `sorted_df` 에 담으세요.
- 정렬 후 맨 위 행의 `연식` 과 `가격만원` 을 확인하세요.

**예시**
```
sorted_df.iloc[0]['연식']    →  2015   (가장 오래된 연식)
sorted_df.iloc[0]['가격만원'] →  4114   (그중 가장 비싼 값)
```

<details><summary>힌트</summary>

```text
접근방법:
- sort_values 에 정렬 기준 열들을 리스트로, 각 열의 오름/내림 여부를 ascending 리스트로 준다.

세부구현:
1. 파일을 df 로 불러온다
2. by 에 ['연식','가격만원'], ascending 에 [True, False] 를 준다
3. 결과를 sorted_df 에 담고 맨 위 행(iloc[0])을 확인한다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
sorted_df = df.sort_values(['연식', '가격만원'], ascending=[True, False])
print(sorted_df.iloc[0]['연식'], sorted_df.iloc[0]['가격만원'])

In [ ]:
# [자가채점]
assert sorted_df.iloc[0]['연식'] == 2015
assert sorted_df.iloc[0]['가격만원'] == 4114
print("✅ 문제11 통과!")

### 해설 — 문제 11
- **접근법**: `by` 에 기준 열들을 순서대로, `ascending` 에 각 열의 방향을 리스트로 맞춰 줍니다. 앞 기준이 우선이에요.
- **흔한 실수**: `ascending` 을 하나만 주면 모든 열에 똑같이 적용됩니다. 열마다 방향이 다르면 **리스트**로 줘야 해요.
- **대안**: 정렬 후 인덱스를 새로 매기려면 `.reset_index(drop=True)` 를 이어 붙입니다.

## 12. 결측값 세어 보기
**배경**: 빈 값(결측)이 어디에 몇 개 있는지 알아야 채우거나 지울지 정할 수 있습니다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- 각 열의 결측 개수를 세어 `na_counts` 에 담으세요. (`isna()` 와 `sum()` 활용, 결과는 Series)

**예시**
```
na_counts['색상']  →  4   (색상 열의 결측 수)
na_counts['모델']  →  0
```

<details><summary>힌트</summary>

```text
접근방법:
- isna() 는 각 칸이 비었는지 참/거짓으로 바꾼다. 열별로 sum() 하면 참(=결측)의 개수가 나온다.

세부구현:
1. 파일을 df 로 불러온다
2. df.isna() 로 비어 있는 칸을 참/거짓 표로 만든다
3. .sum() 으로 열마다 참의 개수를 세어 na_counts 에 담는다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
na_counts = df.isna().sum()
print(na_counts)

In [ ]:
# [자가채점]
assert na_counts['색상'] == 4, "색상 결측 4개"
assert na_counts['모델'] == 0
print("✅ 문제12 통과!")

### 해설 — 문제 12
- **접근법**: `isna()` 는 결측을 `True` 로 표시하고, `sum()` 은 열마다 `True` 의 수(=결측 개수)를 셉니다.
- **흔한 실수**: `df.isna()` 만 하면 참/거짓 표가 나올 뿐입니다. 개수를 세려면 `.sum()` 을 이어 줘야 해요.
- **대안**: 반대로 채워진 값의 수는 `df.notna().sum()`, 결측이 하나라도 있는지는 `df.isna().any()`.

## 13. 결측값 채우기
**배경**: 비어 있는 색상을 지우기 아까울 때는 `'미정'` 같은 값으로 채워 둡니다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `색상` 열의 결측을 문자열 `'미정'` 으로 채운 결과를 `filled` 에 담으세요. (원본 `df` 는 그대로 두어도 됩니다)

**예시**
```
(filled['색상'] == '미정').sum()  →  4
filled['색상'].isna().sum()       →  0
```

<details><summary>힌트</summary>

```text
접근방법:
- fillna 로 결측만 원하는 값으로 바꾼다. 한 열만 채우려면 그 열에 fillna 를 건다.

세부구현:
1. 파일을 df 로 불러온다
2. df.copy() 로 복사본을 만들어 filled 에 담는다
3. filled['색상'] 을 fillna('미정') 결과로 바꾼다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
filled = df.copy()
filled['색상'] = filled['색상'].fillna('미정')
print((filled['색상'] == '미정').sum())

In [ ]:
# [자가채점]
assert (filled['색상'] == '미정').sum() == 4
assert filled['색상'].isna().sum() == 0
print("✅ 문제13 통과!")

### 해설 — 문제 13
- **접근법**: `fillna('미정')` 은 결측만 골라 값을 채웁니다. 결측이 아닌 값은 그대로예요.
- **흔한 실수**: `df.fillna('미정')` 을 df 전체에 걸면 다른 열의 결측까지 바뀔 수 있어요. 한 열만 바꾸려면 그 열에만 겁니다.
- **대안**: 결측 행을 아예 버리려면 `df.dropna(subset=['색상'])` (문제 세트 뒤에서 사용해요).

## 14. 앞·뒤 값으로 결측 채우기 (ffill · bfill)
**배경**: 색상이 빠진 매물을, 값 대신 **바로 앞 행의 색상**으로 채워 봅니다. 순서가 있는 표에서 자주 쓰는 방법입니다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `색상` 의 결측을 **바로 앞 행의 값**으로 채운 결과를 `df2` 에 저장하세요. (`ffill`)
- **원본 `df` 는 그대로 두고**, 채운 결과만 `df2` 에 담으세요. (자가채점이 `df` 가 안 바뀌었는지 함께 확인합니다.)

**예시**
```
df['색상'].isna().sum()   →  4   (채우기 전)
df2['색상'].isna().sum()  →  0   (채운 뒤)
```

<details><summary>힌트</summary>

```text
접근방법:
- ffill 은 바로 앞 값으로, bfill 은 바로 뒤 값으로 빈 칸을 채운다.

세부구현:
1. df 의 색상 열에 .ffill() 을 적용한 결과를 df2 의 색상으로 넣거나, df 를 복사해 채운다
2. 채운 뒤 색상 결측이 0인지 확인
```

</details>

In [ ]:
df = pd.read_csv("../../day06_판다스_기초/data/used_cars.csv")
df2 = df.copy()
df2['색상'] = df2['색상'].ffill()
print(df['색상'].isna().sum(), '→', df2['색상'].isna().sum())

In [ ]:
# [자가채점]
assert df['색상'].isna().sum() == 4
assert df2['색상'].isna().sum() == 0
print("✅ 문제14 통과!")

### 해설 — 문제 14
- **접근법**: `ffill`(forward fill)은 빈 칸을 **바로 앞 행의 값**으로 채웁니다. 뒤 값으로 채우려면 `bfill` 입니다.
- **흔한 실수**: 맨 첫 행이 결측이면 `ffill` 로는 채울 앞 값이 없어 그대로 남습니다(이 데이터는 첫 행이 결측이 아니라 전부 채워짐).
- **대안**: 순서와 무관하게 대표값으로 채우려면 앞 문제처럼 `fillna(값)` 이나 `fillna(중앙값)` 을 씁니다.

## 15. 문자열 숫자를 정수로 바꾸기
**배경**: `주행거리` 는 `"89,000"` 처럼 콤마가 든 문자열이라 계산이 안 됩니다. 콤마를 지우고 정수로 바꿔야 해요.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `주행거리` 의 콤마(`,`)를 지우고 정수형으로 바꿔 `km` 에 담으세요. (`.str.replace` 와 `.astype(int)` 활용)

**예시**
```
km.iloc[0]  →  89000
km.sum()    →  5499999   (콤마를 지운 전체 합)
```

> 참고: `999,999` 처럼 비현실적으로 큰 값(이상치)도 섞여 있어 합이 큽니다. 이상치 처리는 LV2 에서 다룹니다.

<details><summary>힌트</summary>

```text
접근방법:
- 문자열 열의 .str.replace 로 콤마를 빈 문자열로 바꾼 뒤, astype 으로 정수형으로 변환한다.

세부구현:
1. 파일을 df 로 불러온다
2. df['주행거리'].str.replace(',', '') 로 콤마를 없앤다
3. .astype(int) 로 정수형으로 바꿔 km 에 담는다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
km = df['주행거리'].str.replace(',', '').astype(int)
print(km.iloc[0], km.sum())

In [ ]:
# [자가채점]
assert km.iloc[0] == 89000
assert km.sum() == 5499999
assert str(km.dtype).startswith('int')
print("✅ 문제15 통과!")

### 해설 — 문제 15
- **접근법**: `.str.replace(',', '')` 는 문자열 열의 각 값에서 콤마를 지웁니다. 그다음 `.astype(int)` 로 정수형으로 바꿔요.
- **흔한 실수**: `.str` 을 빼먹으면(`df['주행거리'].replace`) 값 전체를 통째로 바꾸려 해 원하는 대로 안 됩니다. 문자열 조작은 `.str` 을 거칩니다.
- **대안**: `regex=False` 를 주면 정규식이 아닌 그냥 콤마로 처리해 더 빠릅니다: `.str.replace(',', '', regex=False)`.

## 16. 이름에 특정 글자가 든 행 찾기
**배경**: "모델명에 '그랜저'가 든 매물"처럼 문자열 일부로 검색할 때 `.str.contains` 를 씁니다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `모델` 에 `'그랜저'` 가 포함된 행만 골라 `grandeur` 에 담으세요.

**예시**
```
len(grandeur)  →  5   (그랜저 매물 수)
```

> 참고: 일부 `모델` 값에는 앞뒤 공백이 있지만, `contains` 는 부분 문자열을 찾으므로 그대로 잡힙니다.

<details><summary>힌트</summary>

```text
접근방법:
- 문자열 열의 .str.contains 로 특정 글자가 들어 있는지 참/거짓을 만들고, 그걸로 행을 거른다.

세부구현:
1. 파일을 df 로 불러온다
2. df['모델'].str.contains('그랜저') 로 조건을 만든다
3. 그 조건을 df[ ... ] 에 넣어 grandeur 를 만든다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')
grandeur = df[df['모델'].str.contains('그랜저')]
print(len(grandeur))

In [ ]:
# [자가채점]
assert len(grandeur) == 5, "그랜저 매물은 5대"
assert grandeur['모델'].str.contains('그랜저').all()
print("✅ 문제16 통과!")

### 해설 — 문제 16
- **접근법**: `.str.contains('그랜저')` 는 값 안에 그 글자가 있으면 `True` 를 줍니다. 앞뒤 공백이 있어도 부분 검색이라 잡혀요.
- **흔한 실수**: 결측이 있는 열에 `contains` 를 걸면 `NaN` 때문에 에러가 날 수 있어요. 그럴 땐 `na=False` 를 줍니다(`모델` 은 결측이 없어 괜찮습니다).
- **대안**: 정확히 일치만 찾으려면 `df['모델'].str.strip() == '그랜저'` 처럼 공백을 지우고 비교합니다.

## 17. 파생 컬럼 만들기 (가격 등급)
**배경**: 원본 값을 바탕으로 새 열을 만들면 분석이 쉬워집니다. 가격을 등급으로 나눠 봅시다.

**요구사항**:
- `data/used_cars.csv` 를 `df` 로 불러오세요.
- `가격만원` 이 **3000 이상**이면 `'고가'`, 아니면 `'저가'` 인 새 열 `가격등급` 을 만드세요. (`apply` 또는 조건식 활용)

**예시**
```
(df['가격등급'] == '고가').sum()  →  20
(df['가격등급'] == '저가').sum()  →  30
```

<details><summary>힌트</summary>

```text
접근방법:
- 가격 한 개를 받아 '고가'/'저가' 를 돌려주는 규칙을 apply 로 모든 행에 적용해 새 열을 만든다.

세부구현:
1. 파일을 df 로 불러온다
2. 값이 3000 이상이면 '고가', 아니면 '저가' 를 돌려주는 규칙을 만든다
3. df['가격만원'].apply(...) 결과를 df['가격등급'] 에 새 열로 넣는다
```

</details>

In [ ]:
import pandas as pd

df = pd.read_csv('../../day06_판다스_기초/data/used_cars.csv')

def price_grade(won):
    if won >= 3000:
        return '고가'
    return '저가'

df['가격등급'] = df['가격만원'].apply(price_grade)
print(df['가격등급'].value_counts())

In [ ]:
# [자가채점]
assert (df['가격등급'] == '고가').sum() == 20
assert (df['가격등급'] == '저가').sum() == 30
print("✅ 문제17 통과!")

### 해설 — 문제 17
- **접근법**: `apply` 는 각 값에 함수를 하나씩 적용합니다. 함수가 규칙(3000 이상이면 고가)을 담고, 결과를 새 열에 대입해요.
- **흔한 실수**: 새 열을 만들 때 `df.가격등급 = ...` 대신 반드시 `df['가격등급'] = ...` (대괄호)로 대입합니다.
- **대안**: `import numpy as np; df['가격등급'] = np.where(df['가격만원'] >= 3000, '고가', '저가')` 로 한 줄로도 만들 수 있어요 (LV2 에서 연습합니다).